# Problem 7

### Problem
Discontinuous boundary conditions on the unit disk $B = B(0; 1)$:
$$
\begin{align*}
\Delta u &= f, \quad \text{in } B = B(0; 1), \\
u &= g, \quad \text{on } \partial B,
\end{align*}
$$
where
$$
f(r e^{i\alpha}) = -4 r^3 (\cos^2\alpha \cdot \sin\alpha + \sin^3\alpha)\sin(1 - r^2) - 8r \sin\alpha \cos(1 - r^2),
$$
and
$$
g(e^{i\alpha}) = \begin{cases}
0, & \alpha \in (0, \pi), \\
1, & \alpha \in (\pi, 2\pi), \\
\frac{1}{2}, & \alpha \in \{0, \pi, 2\pi\}.
\end{cases}
$$
In this case, the exact solution $u$ is given by
$$
u(r e^{i\alpha}) = \frac{1}{2} + \sin(r(1 - r^2)\sin\alpha) - \frac{2}{\pi} \sum_{k=1}^\infty \frac{r^{2k-1} \sin((2k - 1)\alpha)}{2k - 1}, \tag{37}
$$
and in Cartesian coordinates:
$$
f(x, y) = -4(x^2 y + y^3)\sin(1 - x^2 - y^2) - 8y \cos(1 - x^2 - y^2).
$$
The summation in (37) evaluates to machine precision using the closed form:
$$
\sum_{k=1}^\infty \frac{r^{2k-1}\sin((2k-1)\alpha)}{2k-1} = \frac{1}{2} \operatorname{atan2}(2y, \, 1 - x^2 - y^2).
$$

# Imports

In [ ]:
import os
import sys
import platform
import psutil
from pathlib import Path

# ---------------------------------------------------------
# Backend & Environment Configuration
# ---------------------------------------------------------
USE_GPU = False  # Set to True for NVIDIA GPU acceleration

# Auto-detect Google Colab & Auto-Sync
IN_COLAB = 'google.colab' in sys.modules or os.environ.get('COLAB_GPU') is not None
if IN_COLAB:
    if not os.path.exists('/content/NUFFTRR_Poisson'):
        os.system('git clone https://github.com/CharliePyle4/NUFFTRR_Poisson.git /content/NUFFTRR_Poisson')
    
    # Fetch and hard reset to latest GitHub commit
    os.system('cd /content/NUFFTRR_Poisson && git fetch origin main && git reset --hard origin/main')
    
    if os.getcwd() != '/content/NUFFTRR_Poisson':
        os.chdir('/content/NUFFTRR_Poisson')
        
    os.system('pip install -r requirements.txt -q')
    if USE_GPU:
        os.system('pip install cufinufft -q')

# Dynamic repository root and helper path resolution
repo_root = str(Path.cwd().resolve())
while repo_root and not os.path.exists(os.path.join(repo_root, 'Poisson_Solver')):
    parent = str(Path(repo_root).parent)
    if parent == repo_root:
        break
    repo_root = parent
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

tests_dir = os.path.join(repo_root, 'Tests', 'JCP_Paper_Comparisons')
if tests_dir not in sys.path:
    sys.path.insert(0, tests_dir)

# Solver Parameters (Uniform Grid)
num_processors = None  # Number of CPU threads (None = all available cores)

from JCP_Helpers import *

# ---------------------------------------------------------
# System & Hardware Diagnostics Banner
# ---------------------------------------------------------
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
ram_avail_gb = psutil.virtual_memory().available / (1024 ** 3)
cpu_cores_logical = os.cpu_count()
cpu_cores_phys = psutil.cpu_count(logical=False)
cpu_model = platform.processor() or "Generic CPU"
if IN_COLAB and os.path.exists('/proc/cpuinfo'):
    with open('/proc/cpuinfo') as f:
        for line in f:
            if "model name" in line:
                cpu_model = line.split(":", 1)[1].strip()
                break

backend_name = 'GPU (NVIDIA CuPy / cuFFT)' if USE_GPU else 'CPU (Multithreaded FFTW)'

print('=' * 75)
print('  BENCHMARK ENVIRONMENT SUMMARY')
print('=' * 75)
print(f'  Active Backend  : {backend_name}')
print(f'  Environment     : {"Google Colab" if IN_COLAB else "Local Workstation"} ({platform.system()} {platform.release()})')
print(f'  Host CPU        : {cpu_model}')
print(f'  CPU Cores       : {cpu_cores_phys} Physical / {cpu_cores_logical} Logical Threads')
print(f'  System RAM      : {ram_gb:.1f} GB Total ({ram_avail_gb:.1f} GB Available)')

if USE_GPU:
    try:
        import cupy as cp
        dev = cp.cuda.Device()
        props = cp.cuda.runtime.getDeviceProperties(dev.id)
        gpu_name = props['name'].decode('utf-8') if isinstance(props['name'], bytes) else str(props['name'])
        compute_cap = f"{props['major']}.{props['minor']}"
        free_mem_gb, total_mem_gb = dev.mem_info[0] / (1024**3), dev.mem_info[1] / (1024**3)
        
        print('-' * 75)
        print(f'  GPU Device      : {gpu_name} (Compute Capability: {compute_cap})')
        print(f'  GPU VRAM        : {total_mem_gb:.1f} GB Total ({free_mem_gb:.1f} GB Free)')
        print(f'  CUDA Driver/RT  : {cp.cuda.runtime.runtimeGetVersion()}')
    except Exception as e:
        print(f'  GPU Status      : Warning - GPU query failed: {e}')

print('=' * 75)

# Problem Setup

In [ ]:
# Problem Setup — Problem 7 (Borges–Daripa JCP)

u, f, g_dirichlet, g_neumann = setup_problem_7()

# Radius
R = 1.0

# Radial mesh type: uniform
rad_unif = 1

# Methods to test
methods = [
    dict(
        name="uniform_fft",
        label="Uniform Mesh",
        azu_unif=2,
        mesh_kind=None,
        use_nudft=None,
    ),
]

BC_MAP = {
    "dirichlet": 1,
    "neumann": 2,
}

QUAD_MAP = {
    "trapezoidal": 1,
    "simpson": 2,
}

# N and M values for Table X
N_values = [64, 128, 256, 512, 1024, 2048]
M_values = [64, 128, 256, 512, 1024, 2048]

# Table X: Relative Errors in Norm $\|\cdot\|_\infty$
Relative errors evaluated over the masked domain $B(0; 1) - (B_{0.01}(1, 0) \cup B_{0.01}(-1, 0))$.

In [ ]:
# Run Table X: Relative Errors in Norm ||·||_∞
df_table10 = run_table_10(methods, N_values, M_values, u, f, g_dirichlet, g_neumann, BC_MAP, QUAD_MAP, rad_unif, R, mask_radius=0.01)
display_table_10(df_table10, methods, N_values, M_values)

# Visualizations: Figures 11–14
- **Figure 11**: Problem 7—Analytical solution $u$.
- **Figure 12**: Problem 7—Errors for 64 Fourier coefficients and 256 circles ($N=64, M=256$).
- **Figure 13**: Problem 7—Errors for 128 Fourier coefficients and 256 circles ($N=128, M=256$).
- **Figure 14**: Problem 7—Errors for 256 Fourier coefficients and 256 circles ($N=256, M=256$).

In [ ]:
M_fig = 256
method_fig = methods[0]

# Solve for N = 64
x_coord_64, y_coord_64, u_approx_64, u_true_64 = solve_for_grids(
    64, M_fig, method_fig, "dirichlet", "trapezoidal",
    u, f, g_dirichlet, g_neumann, BC_MAP, QUAD_MAP, rad_unif, R
)

# Solve for N = 128
x_coord_128, y_coord_128, u_approx_128, u_true_128 = solve_for_grids(
    128, M_fig, method_fig, "dirichlet", "trapezoidal",
    u, f, g_dirichlet, g_neumann, BC_MAP, QUAD_MAP, rad_unif, R
)

# Solve for N = 256
x_coord_256, y_coord_256, u_approx_256, u_true_256 = solve_for_grids(
    256, M_fig, method_fig, "dirichlet", "trapezoidal",
    u, f, g_dirichlet, g_neumann, BC_MAP, QUAD_MAP, rad_unif, R
)

In [ ]:
# Figure 11: Analytical solution u
plot_on_disk(x_coord_256, y_coord_256, u_true_256, title="Figure 11: Problem 7—Analytical solution")

# Figure 12: Errors for N = 64, M = 256
plot_on_disk(x_coord_64, y_coord_64, np.abs(u_true_64 - u_approx_64), title="Figure 12: Problem 7—Errors (N=64, M=256)")

# Figure 13: Errors for N = 128, M = 256
plot_on_disk(x_coord_128, y_coord_128, np.abs(u_true_128 - u_approx_128), title="Figure 13: Problem 7—Errors (N=128, M=256)")

# Figure 14: Errors for N = 256, M = 256
plot_on_disk(x_coord_256, y_coord_256, np.abs(u_true_256 - u_approx_256), title="Figure 14: Problem 7—Errors (N=256, M=256)")

## One-Dimensional Section Errors: Figure 15
Errors when considering the one-dimensional section of the disk $B(0; 1)$ laying on the segment from $(0, -1)$ to $(0, 1)$ along the vertical diameter (radial position $-1$ at $(0, -1)$, and $+1$ at $(0, 1)$):
- **(a)** Linear plot showing convergence as the number of Fourier coefficients increases from 64 to 128 and 256.
- **(b)** Log-scaling plot showing the rate of convergence.

In [ ]:
# Figure 15: 1D section errors along (0, -1) to (0, 1)
solutions_fig15 = {
    64: (x_coord_64, y_coord_64, u_approx_64, u_true_64),
    128: (x_coord_128, y_coord_128, u_approx_128, u_true_128),
    256: (x_coord_256, y_coord_256, u_approx_256, u_true_256),
}
plot_problem_7_1d_section(solutions_fig15, R=R)